In [1]:
import numpy as np
import pandas as pd
import joblib

Load trained model + feature order

In [ ]:
model = joblib.load("models/activity_classifier_rf.pkl")
feature_columns = joblib.load("models/feature_columns.pkl")

print("Model loaded.")
print(feature_columns)

Define SAME feature functions

In [ ]:
def vector_magnitude(x, y, z):
    return np.sqrt(x**2 + y**2 + z**2)

def extract_imu_features(ax, ay, az, gx, gy, gz):
    acc_mag = vector_magnitude(ax, ay, az)
    gyro_mag = vector_magnitude(gx, gy, gz)

    return {
        "acc_mean": np.mean(acc_mag),
        "acc_std": np.std(acc_mag),
        "acc_energy": np.mean(acc_mag**2),
        "gyro_mean": np.mean(gyro_mag),
        "gyro_std": np.std(gyro_mag)
    }

def extract_hr_features(hr):
    return {
        "hr_mean": np.mean(hr),
        "hr_std": np.std(hr)
    }

def extract_temp_features(temp):
    return {
        "temp_synth": np.mean(temp)   # rename later to temp_mean if you want
    }

def extract_emg_features(emg):
    return {
        "emg_rms_synth": np.sqrt(np.mean(emg**2)),
        "emg_mav_synth": np.mean(np.abs(emg)),
        "emg_wl_synth": np.sum(np.abs(np.diff(emg)))
    }

Load real sensor data

In [ ]:
imu_df = pd.read_csv("data/sensors/imu.csv")
hr_df = pd.read_csv("data/sensors/hr.csv")
temp_df = pd.read_csv("data/sensors/temp.csv")
emg_df = pd.read_csv("data/sensors/emg.csv")

Window extraction

In [ ]:
WINDOW_SIZE = 200

imu_window = imu_df.iloc[-WINDOW_SIZE:]
hr_window = hr_df.iloc[-WINDOW_SIZE:]
temp_window = temp_df.iloc[-WINDOW_SIZE:]
emg_window = emg_df.iloc[-WINDOW_SIZE:]

Extract features

In [ ]:
imu_features = extract_imu_features(
    imu_window["ax"].values,
    imu_window["ay"].values,
    imu_window["az"].values,
    imu_window["gx"].values,
    imu_window["gy"].values,
    imu_window["gz"].values
)

hr_features = extract_hr_features(hr_window["hr"].values)
temp_features = extract_temp_features(temp_window["temp"].values)
emg_features = extract_emg_features(emg_window["emg"].values)

Merge features

In [ ]:
features = {}
features.update(hr_features)
features.update(imu_features)
features.update(temp_features)
features.update(emg_features)

features_df = pd.DataFrame([features])

# ensure correct order
features_df = features_df[feature_columns]

features_df

Predict

In [ ]:
prediction = model.predict(features_df)[0]

label_map = {
    0: "Rest",
    1: "Physical Activity",
    2: "Intense Activity"
}

print("Prediction:", label_map[prediction])

Probablity prediction

In [ ]:
proba = model.predict_proba(features_df)[0]

for i, p in enumerate(proba):
    print(label_map[i], ":", round(p, 3))

Wrap into function

In [ ]:
def predict_activity(imu_df, hr_df, temp_df, emg_df):
    imu_window = imu_df.iloc[-WINDOW_SIZE:]
    hr_window = hr_df.iloc[-WINDOW_SIZE:]
    temp_window = temp_df.iloc[-WINDOW_SIZE:]
    emg_window = emg_df.iloc[-WINDOW_SIZE:]

    imu_features = extract_imu_features(
        imu_window["ax"].values,
        imu_window["ay"].values,
        imu_window["az"].values,
        imu_window["gx"].values,
        imu_window["gy"].values,
        imu_window["gz"].values
    )

    hr_features = extract_hr_features(hr_window["hr"].values)
    temp_features = extract_temp_features(temp_window["temp"].values)
    emg_features = extract_emg_features(emg_window["emg"].values)

    features = {}
    features.update(hr_features)
    features.update(imu_features)
    features.update(temp_features)
    features.update(emg_features)

    features_df = pd.DataFrame([features])
    features_df = features_df[feature_columns]

    pred = model.predict(features_df)[0]
    return label_map[pred]